# 5. LLM Integration Pipeline

Integrates the **IDS model, SHAP, RAG, and LLM** to generate concise cybersecurity incident reports.


## 1. Import Libraries


In [72]:
import joblib
import numpy as np
import pandas as pd
import shap
from openai import OpenAI
import os

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

## 2. Load Machine Learning Assets

In [73]:
model = joblib.load("models/rf_model.pkl")
scaler = joblib.load("models/scaler.pkl")
label_encoder = joblib.load("models/label_encoder.pkl")
feature_names = joblib.load("artifacts/feature_names.pkl")

## 3. Load RAG Components

In [74]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}
)

vector_db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 4. Load Test Data

In [75]:
# Load Test Dataset

X_test = joblib.load("X_test_selected.pkl")
X_test_z = joblib.load("X_test_z.pkl")
y_test_encoded = joblib.load("y_test_encoded.pkl")

print(f"Test samples : {X_test_z.shape[0]}")
print(f"Features     : {X_test_z.shape[1]}")

Test samples : 374729
Features     : 17


## 5. Select Network Traffic Sample

In [76]:
sample_index = 0

sample = X_test_z.iloc[[sample_index]]        
sample_raw = X_test.iloc[[sample_index]]     
print("Scaled sample:")
print(sample)

print("\nRaw sample:")
print(sample_raw)

Scaled sample:
         Idle Mean_log  Idle Max_log  Bwd Packet Length Std_log  \
2431708       2.018145      2.011547                   1.879159   

         Bwd Packet Length Mean_log  Flow Bytes/s  \
2431708                    1.491164     -0.052867   

         Total Length of Fwd Packets_log  Fwd Packet Length Max_log  \
2431708                         0.777857                   1.072589   

         act_data_pkt_fwd_log  Average Packet Size_log  \
2431708              -0.18087                 1.403787   

         Fwd Packet Length Std_log  Max Packet Length_log  \
2431708                   1.378799               1.544389   

         Packet_Variation_Ratio  Flow Packets/s  Packet_Size_Consistency  \
2431708               -0.259505       -0.233212                -0.042603   

         Idle_Activity_Ratio  Min Packet Length_log  Bwd Header Length  
2431708             1.839267              -0.938878           0.001841  

Raw sample:
         Idle Max_log  Flow Packets/s  Bwd Packe

In [77]:
idle_max_log_value = sample_raw['Idle Max_log'].values[0]  # 18.26753

idle_max_original = np.expm1(idle_max_log_value)

## 6. Predict Attack Type

Predict the attack class using the trained Random Forest model and calculate the prediction confidence.

In [78]:
pred_encoded = model.predict(sample)[0]

probabilities = model.predict_proba(sample)[0]

attack = label_encoder.inverse_transform([pred_encoded])[0]

confidence = probabilities[pred_encoded] * 100

print(f"Predicted Attack : {attack}")
print(f"Confidence Score : {confidence:.2f}%")

Predicted Attack : DoS
Confidence Score : 100.00%


## 7. Explain the Prediction Using SHAP

Generate SHAP values for the predicted sample to identify the most influential features contributing to the model's decision.

In [79]:
explainer = shap.TreeExplainer(model)

In [80]:
# Compute SHAP Values

shap_values = explainer.shap_values(sample)

print(type(shap_values))
print(shap_values.shape)

<class 'numpy.ndarray'>
(1, 17, 8)


In [81]:
pred_idx = pred_encoded

class_shap_values = shap_values[0, :, pred_idx]

In [82]:
print(impact_df["Feature"].tolist())
print(sample_raw.columns.tolist())

['Bwd Packet Length Std_log', 'Bwd Packet Length Mean_log', 'Max Packet Length_log', 'Idle Mean_log', 'Average Packet Size_log', 'Idle Max_log', 'Fwd Packet Length Max_log', 'Flow Packets/s', 'Fwd Packet Length Std_log', 'Total Length of Fwd Packets_log', 'Min Packet Length_log', 'Bwd Header Length', 'act_data_pkt_fwd_log', 'Flow Bytes/s', 'Idle_Activity_Ratio', 'Packet_Size_Consistency', 'Packet_Variation_Ratio']
['Idle Max_log', 'Flow Packets/s', 'Bwd Packet Length Std_log', 'Idle Mean_log', 'Flow Bytes/s', 'Bwd Packet Length Mean_log', 'Fwd PSH Flags', 'Total Length of Fwd Packets_log', 'Fwd Packet Length Std_log', 'act_data_pkt_fwd_log', 'Average Packet Size_log', 'Fwd Packet Length Max_log', 'Bwd IAT Total_log', 'Max Packet Length_log', 'Bwd Header Length', 'Fwd Packet Length Mean_log', 'Min Packet Length_log', 'Bytes_Per_Packet', 'Packet_Size_Consistency', 'Idle_Activity_Ratio', 'Fwd_Packet_Profile', 'Packet_Variation_Ratio']


In [83]:
print("MODEL FEATURES:")
print(feature_names)

print("\nX_TEST_Z FEATURES:")
print(X_test_z.columns.tolist())

print("\nMODEL FEATURE COUNT:", len(feature_names))
print("X_TEST_Z FEATURE COUNT:", len(X_test_z.columns))

MODEL FEATURES:
['Idle Mean_log', 'Idle Max_log', 'Bwd Packet Length Std_log', 'Bwd Packet Length Mean_log', 'Flow Bytes/s', 'Total Length of Fwd Packets_log', 'Fwd Packet Length Max_log', 'act_data_pkt_fwd_log', 'Average Packet Size_log', 'Fwd Packet Length Std_log', 'Max Packet Length_log', 'Packet_Variation_Ratio', 'Flow Packets/s', 'Packet_Size_Consistency', 'Idle_Activity_Ratio', 'Min Packet Length_log', 'Bwd Header Length']

X_TEST_Z FEATURES:
['Idle Mean_log', 'Idle Max_log', 'Bwd Packet Length Std_log', 'Bwd Packet Length Mean_log', 'Flow Bytes/s', 'Total Length of Fwd Packets_log', 'Fwd Packet Length Max_log', 'act_data_pkt_fwd_log', 'Average Packet Size_log', 'Fwd Packet Length Std_log', 'Max Packet Length_log', 'Packet_Variation_Ratio', 'Flow Packets/s', 'Packet_Size_Consistency', 'Idle_Activity_Ratio', 'Min Packet Length_log', 'Bwd Header Length']

MODEL FEATURE COUNT: 17
X_TEST_Z FEATURE COUNT: 17


In [84]:
# Create SHAP Explanation Table

impact_df = pd.DataFrame({
    "Feature": sample.columns,
    "Scaled_Value": sample.iloc[0].values,
    "SHAP": class_shap_values
})

# SHAP Contribution Direction
impact_df["Direction"] = np.where(
    impact_df["SHAP"] >= 0,
    "Positive",
    "Negative"
)

# SHAP Importance
impact_df["Abs_SHAP"] = impact_df["SHAP"].abs()

# Rank Features
impact_df = impact_df.sort_values(
    "Abs_SHAP",
    ascending=False
).reset_index(drop=True)

impact_df["Rank"] = range(1, len(impact_df) + 1)

# Impact Level
impact_df["Impact"] = np.select(
    [
        impact_df["Rank"] <= 2,
        impact_df["Rank"].between(3, 4),
        impact_df["Rank"] >= 5
    ],
    [
        "High",
        "Medium",
        "Low"
    ],
    default="Low"
)


# Recover Original Feature Values

def get_observed_value(feature_name):
    value = sample_raw[feature_name].iloc[0]

    # Reverse log1p transformation
    if feature_name.endswith("_log"):
        return np.expm1(value)

    return value


impact_df["Observed_Value"] = impact_df["Feature"].apply(
    get_observed_value
)


# Display Feature Name
impact_df["Display_Feature"] = (
    impact_df["Feature"]
    .str.replace("_log", "", regex=False)
)


# Top 5 Features

top_features = (
    impact_df[
        [
            "Rank",
            "Display_Feature",
            "Observed_Value",
            "SHAP",
            "Direction",
            "Impact"
        ]
    ]
    .head(5)
    .reset_index(drop=True)
)


print("Top 5 Features Influencing the Prediction:")
print(top_features)

Top 5 Features Influencing the Prediction:
   Rank         Display_Feature  Observed_Value      SHAP Direction  Impact
0     1   Bwd Packet Length Std    2.537820e+03  0.258676  Positive    High
1     2  Bwd Packet Length Mean    1.932500e+03  0.131569  Positive    High
2     3       Max Packet Length    5.792000e+03  0.089916  Positive  Medium
3     4               Idle Mean    8.580007e+07  0.055976  Positive  Medium
4     5     Average Packet Size    9.183848e+02  0.055958  Positive     Low


## 8. Retrieve Relevant Knowledge from the RAG Database

Retrieve cybersecurity knowledge related to the predicted attack from the FAISS vector database. The retrieved context will provide technical information such as attack description, indicators of compromise (IoCs), detection methods, MITRE ATT&CK mapping, mitigation strategies, and incident response recommendations.

In [85]:
# Retrieval Query

query = (
    f"{attack} attack description, indicators of compromise, "
    f"detection methods, MITRE ATT&CK mapping, "
    f"recommended mitigation, and incident response"
)

In [86]:
results = vector_db.similarity_search(
    query,
    k=2,
    filter={"attack": attack}
)

In [87]:
for i, doc in enumerate(results, 1):

    print("="*80)

    print(f"Document {i}")

    print(f"Attack : {doc.metadata['attack']}")

    print(f"Source : {doc.metadata['source']}")

    print()

    print(doc.page_content[:500])

    print()

Document 1
Attack : DoS
Source : ..\knowledge_base\DoS.md

## Indicators of Compromise (IoCs)
- Network: high request rate or abnormally long-held connections from a single IP.
- Host: elevated CPU/memory/connection table usage tied to one source.
- Behavioral: service degradation correlating with sustained single-source activity.
- Log-based: web/application server logs showing repeated requests or incomplete request patterns from one origin.

## MITRE ATT&CK Mapping
- Tactic: Impact — Technique: Endpoint Denial of Service (T1499), including sub-techni

Document 2
Attack : DoS
Source : ..\knowledge_base\DoS.md

## SOC Analyst Investigation Guide
Identify the single source IP and request pattern, correlate with server resource metrics, determine attack subtype (flood vs. slow-rate), and confirm whether the target application has known DoS-relevant misconfigurations.

## Common False Positives
Misbehaving legitimate clients (retry loops, broken automation scripts) can generate DoS-like 

## 9. Build Structured Prompt

Construct a structured prompt by combining the machine learning prediction, confidence score, SHAP explanation, and retrieved cybersecurity knowledge. This prompt provides the Large Language Model (LLM) with both quantitative evidence and domain-specific context for generating an accurate AI-powered incident report.

In [88]:
from datetime import datetime

report_id = f"IR-{datetime.now():%Y%m%d-%H%M%S}"

report_date = datetime.now().strftime("%Y-%m-%d")

In [89]:
# Build SHAP Summary

shap_summary = ""

for _, row in top_features.iterrows():

    shap_summary += f"""
Rank: {row['Rank']}
Feature: {row['Display_Feature']}
Observed Value: {row['Observed_Value']:.2f}
SHAP Contribution: {row['SHAP']:.4f}
Contribution Direction: {row['Direction']}
Impact Level: {row['Impact']}

"""

print(shap_summary)


Rank: 1
Feature: Bwd Packet Length Std
Observed Value: 2537.82
SHAP Contribution: 0.2587
Contribution Direction: Positive
Impact Level: High


Rank: 2
Feature: Bwd Packet Length Mean
Observed Value: 1932.50
SHAP Contribution: 0.1316
Contribution Direction: Positive
Impact Level: High


Rank: 3
Feature: Max Packet Length
Observed Value: 5792.00
SHAP Contribution: 0.0899
Contribution Direction: Positive
Impact Level: Medium


Rank: 4
Feature: Idle Mean
Observed Value: 85800072.00
SHAP Contribution: 0.0560
Contribution Direction: Positive
Impact Level: Medium


Rank: 5
Feature: Average Packet Size
Observed Value: 918.38
SHAP Contribution: 0.0560
Contribution Direction: Positive
Impact Level: Low




In [90]:
import re

# Build Retrieved Context

def clean_content(text):
    """Collapse multiple blank lines and strip leading/trailing whitespace."""
    text = text.strip()
    return re.sub(r"\n{2,}", "\n\n", text)


context_parts = []

for i, doc in enumerate(results, start=1):
    attack_name = doc.metadata.get("attack", "Unknown")
    source = doc.metadata.get("source", "Unknown")
    content = clean_content(doc.page_content)

    context_parts.append(
        f"[Document {i} | Attack: {attack_name} | Source: {source}]\n{content}"
    )

context = "\n\n".join(context_parts)

print(f"[INFO] Built context from {len(results)} retrieved document(s), "
      f"{len(context)} characters total.")

[INFO] Built context from 2 retrieved document(s), 1481 characters total.


In [91]:
# Build Prompt

prompt = f"""
You are a Senior SOC Analyst, Threat Hunter, and Incident Response Specialist.

Generate a professional, evidence-based cybersecurity incident report in Markdown.

Your report MUST rely exclusively on the following evidence:

1. Machine Learning prediction
2. SHAP explainability results
3. Retrieved cybersecurity knowledge (RAG)

Do not use external cybersecurity knowledge.
Do not fabricate technical details.
If the supplied evidence is insufficient, clearly state that additional investigation is required.

---

# Incident Metadata

Report ID: {report_id}

Report Date: {report_date}

Attack Type: {attack}

Prediction Confidence: {confidence:.2f}%

Machine Learning Model: Random Forest

Dataset: CICIDS2017

Explainability Method: SHAP

Knowledge Base: FAISS Retrieval-Augmented Generation (RAG)

---

# SHAP Explainability Results

{shap_summary}

---

# Retrieved Cybersecurity Knowledge

{context}

---

# Required Report Structure

## Executive Summary

Summarize:

- suspected attack
- potential operational impact
- analyst recommendation

Do not describe the incident as confirmed.

---

## Attack Characteristics

Present as a Markdown table containing:

- Attack Type
- Prediction Confidence
- Machine Learning Model
- Dataset
- Explainability Method
- Knowledge Base

---

## Evidence Sources

Present as a Markdown table.

Include:

- Prediction → Random Forest
- Confidence → Random Forest
- Explainability → SHAP
- Threat Intelligence → FAISS RAG

---

## Attack Description

Describe the observed attack behavior using ONLY the retrieved cybersecurity knowledge.

---

## Prediction Confidence Assessment

Explain that the confidence score represents the Random Forest model's prediction confidence.

Clearly state that it is NOT proof of malicious activity.

Recommend analyst verification before containment.

---

## Explainable AI Analysis (SHAP)

Interpret every SHAP feature separately.

For each feature include:

- Feature Name
- Observed Feature Value
- SHAP Contribution
- Contribution Direction
- Impact Level
- Explain ONLY how the feature influenced the machine learning prediction.

Never interpret SHAP values as proof of an attack.

---

## Indicators of Compromise (IoCs)

Summarize ONLY the IoCs contained in the retrieved knowledge.

---

## MITRE ATT&CK Mapping

Include ONLY mappings explicitly present in the retrieved knowledge.

---

## Recommended Mitigation

Summarize ONLY the mitigation recommendations from the retrieved knowledge.

---

## Incident Response Actions

Provide practical SOC actions including:

- validation
- log analysis
- containment
- monitoring
- evidence collection

---

## False Positive Considerations

Summarize any legitimate scenarios described in the retrieved knowledge.

If unavailable, recommend analyst validation.

---

## Risk Assessment

Estimate:

- Likelihood
- Business Impact
- Overall Risk Level

Base the assessment ONLY on the supplied evidence.

---

## SOC Analyst Conclusion

Combine:

- Machine Learning prediction
- Prediction confidence
- SHAP explainability
- Retrieved cybersecurity knowledge

Provide a concise recommendation for the SOC team.

---

# Writing Rules

- Professional Markdown.
- Concise SOC writing style.
- Evidence-based conclusions only.
- Never fabricate IoCs, MITRE techniques, mitigations, or attack behavior.
- Never claim that prediction confidence confirms an attack.
- Never infer causation from SHAP values.
- Explain SHAP only as model explainability.
- Integrate retrieved knowledge naturally instead of copying it verbatim.
- Keep the report concise, technical, and suitable for SOC documentation.
"""

In [92]:
print(prompt[:2500])


You are a Senior SOC Analyst, Threat Hunter, and Incident Response Specialist.

Generate a professional, evidence-based cybersecurity incident report in Markdown.

Your report MUST rely exclusively on the following evidence:

1. Machine Learning prediction
2. SHAP explainability results
3. Retrieved cybersecurity knowledge (RAG)

Do not use external cybersecurity knowledge.
Do not fabricate technical details.
If the supplied evidence is insufficient, clearly state that additional investigation is required.

---

# Incident Metadata

Report ID: IR-20260809-041900

Report Date: 2026-08-09

Attack Type: DoS

Prediction Confidence: 100.00%

Machine Learning Model: Random Forest

Dataset: CICIDS2017

Explainability Method: SHAP

Knowledge Base: FAISS Retrieval-Augmented Generation (RAG)

---

# SHAP Explainability Results


Rank: 1
Feature: Bwd Packet Length Std
Observed Value: 2537.82
SHAP Contribution: 0.2587
Contribution Direction: Positive
Impact Level: High


Rank: 2
Feature: Bwd Pack

## 10. Generate AI Incident Report Using LLM

Send the structured prompt to a Large Language Model (LLM) to generate a professional cybersecurity incident report. The report is generated using the machine learning prediction, SHAP explanation, and retrieved knowledge from the RAG knowledge base.

In [93]:
from google import genai

In [94]:
from dotenv import load_dotenv


In [95]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [96]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=prompt
)

incident_report = response.text

print(incident_report)

# Cybersecurity Incident Report: Suspected Denial of Service (DoS) Activity

## Incident Metadata
* **Report ID:** IR-20260809-041900
* **Report Date:** 2026-08-09
* **Target Dataset:** CICIDS2017

---

## Executive Summary

The Security Operations Center (SOC) detected potential anomalous traffic patterns categorized by the machine learning detection engine as a **suspected Denial of Service (DoS)** attempt. 

If validated, the operational impact could include service degradation or operational downtime due to single-source resource exhaustion targeting the destination service. At present, this incident is treated as **suspected** and requires analyst validation. The primary analyst recommendation is to manually verify target server resource metrics and web/application logs to confirm source intent prior to executing containment actions.

---

## Attack Characteristics

| Parameter | Value |
| :--- | :--- |
| **Attack Type** | DoS |
| **Prediction Confidence** | 100.00% |
| **Machine 

In [97]:
# Save AI Incident Report

with open("AI_Incident_Report.md", "w", encoding="utf-8") as file:
    file.write(incident_report)

print("AI Incident Report saved successfully.")

AI Incident Report saved successfully.


## 11. Final AI Incident Report

In [98]:
import os
import joblib

os.makedirs("artifacts", exist_ok=True)

# Model
joblib.dump(model, "artifacts/rf_model.pkl")

# Preprocessing
joblib.dump(scaler, "artifacts/scaler.pkl")
joblib.dump(label_encoder, "artifacts/label_encoder.pkl")
joblib.dump(feature_names, "artifacts/feature_names.pkl")

# Test Data
joblib.dump(X_test, "artifacts/X_test.pkl")
joblib.dump(X_test_z, "artifacts/X_test_z.pkl")

y_test = label_encoder.inverse_transform(y_test_encoded)
joblib.dump(y_test, "artifacts/y_test.pkl")
joblib.dump(y_test_encoded, "artifacts/y_test_encoded.pkl")

['artifacts/y_test_encoded.pkl']